In [2]:
#Import Libraries
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sb 
import pyodbc
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

In [3]:
##1. Connect Database/data
connection_url = URL.create(
    "mssql+pyodbc",
    host=r"(local)\SQLEXPRESS",
    database="Churn_Customers",
    query={
        "driver": "ODBC Driver 17 for SQL Server",
        "trusted_connection": "yes"
    }
)

engine = create_engine(connection_url)

try:
    with engine.connect() as conn:
        print("Connected successfully!")
except Exception as e:
    print(e)

Connected successfully!


In [4]:
df = pd.read_sql(
    "SELECT * FROM db_customer",  #dbo.db_customer
    engine
)

df.head()

,customerid,name,country,state,gender,dob,interests,pincode
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12,travel,None
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23,None,None
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15,movie,None
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30,None,None
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05,drama,None


In [5]:
sql_query = """SELECT name
            FROM sys.objects
            WHERE type = 'U';"""
            
tables = pd.read_sql(sql_query, engine)

#Create Dataframe from each table
for table_name in tables['name']:
    df = pd.read_sql(f"select * from {table_name}" , engine)
    globals()[f"df_{table_name}"] = df
    print(f"Created dataframe: df_{table_name}")

Created dataframe: df_db_customer
Created dataframe: df_db_subscription
Created dataframe: df_db_support


In [6]:
# Check the datasets for the consistency of data
df_db_customer.head()
df_db_subscription.head()
df_db_support.head()

#Close the SQL Engine
engine.dispose()

In [7]:
#Checking table Schema
for table_name in tables['name']:
    print(f"Table Name: {table_name}")
    print(df.columns)

Table Name: db_customer
Index(['customerid', 'complaint_date', 'escalations', 'csat_score', 'col_1',
       'comment'],
      dtype='object')
Table Name: db_subscription
Index(['customerid', 'complaint_date', 'escalations', 'csat_score', 'col_1',
       'comment'],
      dtype='object')
Table Name: db_support
Index(['customerid', 'complaint_date', 'escalations', 'csat_score', 'col_1',
       'comment'],
      dtype='object')


In [8]:
##Data Cleaning - df_db_customer
df_db_customer.info()
df_db_customer.isnull().sum()
df_db_customer.isnull()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   customerid  21 non-null     object        
 1   name        21 non-null     object        
 2   country     18 non-null     object        
 3   state       21 non-null     object        
 4   gender      21 non-null     object        
 5   dob         21 non-null     datetime64[ns]
 6   interests   4 non-null      object        
 7   pincode     0 non-null      object        
dtypes: datetime64[ns](1), object(7)
memory usage: 1.4+ KB


,customerid,name,country,state,gender,dob,interests,pincode
0,False,False,False,False,False,False,False,True
1,False,False,False,False,False,False,True,True
2,False,False,False,False,False,False,False,True
3,False,False,False,False,False,False,True,True
4,False,False,False,False,False,False,False,True
5,False,False,True,False,False,False,True,True
6,False,False,False,False,False,False,True,True
7,False,False,False,False,False,False,True,True
8,False,False,True,False,False,False,True,True
9,False,False,False,False,False,False,True,True


In [9]:
#a. rename col - name
#b. drop columns - interest and pincode
#c. change datatype of DOB
#d. data standardization -gender
#e. Fix missing values in Countries

In [10]:
#a. rename col - name
df_db_customer.rename(columns= {'name' : 'customer_name'}, inplace=True)
df_db_customer.head(2)

,customerid,customer_name,country,state,gender,dob,interests,pincode
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12,travel,None
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23,None,None


In [11]:
#b. drop columns - interest and pincode
# can drop by index value as the last 2 columns are to be deleted
# df_db_customer.columns[6:8]
# df_db_customer.columns[-4:5]
df_db_customer.drop(df_db_customer.columns[-2:], axis=1, inplace=True)

# can also delete by direct column name
# df_db_customer.drop(columns=['interests', 'pincode'])

In [12]:
df_db_customer.head(2)

,customerid,customer_name,country,state,gender,dob
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23


In [13]:
#c. change datatype of DOB
df_db_customer['dob'].info()

pd.to_datetime(df_db_customer['dob'])

<class 'pandas.core.series.Series'>
RangeIndex: 21 entries, 0 to 20
Series name: dob
Non-Null Count  Dtype         
--------------  -----         
21 non-null     datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 300.0 bytes


0    1982-04-12
1    1995-11-23
2    1978-02-15
3    2001-08-30
4    1990-05-05
5    1988-12-10
6    1976-09-21
7    1999-03-14
8    1985-07-07
9    1993-10-29
10   1997-01-22
11   1981-06-18
12   2004-12-01
13   1992-04-25
14   1979-11-11
15   1986-02-28
16   1994-08-19
17   2000-09-02
18   1983-12-30
19   1991-05-14
20   1977-10-06
Name: dob, dtype: datetime64[ns]

In [14]:
#d. data standardization -gender
df_db_customer['gender'].head()
df_db_customer['gender'].unique()  #Make all to either 'Male' or 'Female'
df_db_customer['gender'].replace({'Men' : 'Male', 'Women' : 'Female'}, inplace=True)
df_db_customer['gender'].unique()

C:\Users\dibya\AppData\Local\Temp\ipykernel_5408\1693658674.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_db_customer['gender'].replace({'Men' : 'Male', 'Women' : 'Female'}, inplace=True)


array(['Male', 'Female'], dtype=object)

In [15]:
#e. Fix missing values in Countries
df_db_customer['country'].isna().sum()
df_db_customer[df_db_customer['country'].isna()]  #This works on basis of Index

,customerid,customer_name,country,state,gender,dob
5,0013-MHZWF,durga,None,Delhi,Female,1988-12-10
8,0015-UOCOJ,maya,None,Kathmandu,Female,1985-07-07
12,0018-NYROU,chitra,None,Telangana,Female,2004-12-01


In [16]:
df_db_customer['country'].fillna('Nahi Pata')
df_db_customer[['country', 'state']]

#Fill by matching with existing data, else can go by mobile country code
state_country_mapping = df_db_customer.dropna(subset=['country']).set_index('state')['country'].to_dict()
df_db_customer['country'] = df_db_customer['country'].fillna(df_db_customer['state'].map(state_country_mapping))
df_db_customer['country'].isna().sum()

0

In [17]:
df_db_subscription.head()
df_db_subscription.info() #cltv: customer life time value 

date_col = ['subscription_start_date', 'renewal_date', 'cancellation_date']
df_db_subscription[date_col] = df_db_subscription[date_col].apply(pd.to_datetime)

df_db_subscription.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customerid               21 non-null     object 
 1   subscription_start_date  21 non-null     object 
 2   subscription_type        21 non-null     object 
 3   renewal_date             21 non-null     object 
 4   plan_type                21 non-null     object 
 5   contract_type            21 non-null     object 
 6   cancellation_date        6 non-null      object 
 7   cancellation_reason      6 non-null      object 
 8   monthly_charges          21 non-null     float64
 9   cltv                     21 non-null     int64  
 10  churn_score              21 non-null     int64  
dtypes: float64(1), int64(2), object(8)
memory usage: 1.9+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 11 columns):
 #   Column                   

In [18]:
df_db_support.head()
df_db_support.drop(columns=['col_1'], inplace=True)

In [19]:
df_db_support['complaint_date'] = pd.to_datetime(df_db_support['complaint_date'])
df_db_support['complaint_date'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 9 entries, 0 to 8
Series name: complaint_date
Non-Null Count  Dtype         
--------------  -----         
9 non-null      datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 204.0 bytes


In [20]:
df.to_csv('CustomerChurn_Clean.csv', index=False)

In [23]:
# Use ExcelWriter to write to a single file
with pd.ExcelWriter('CustomerChurn_Clean.xlsx', engine='openpyxl') as writer:
    df_db_customer.to_excel(writer, sheet_name='Customers', index=False)
    df_db_subscription.to_excel(writer, sheet_name='Subscription', index=False)
    df_db_support.to_excel(writer, sheet_name='Support', index=False)